# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}, Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant datasets, each record set, field, and column is referenced by its unique `@id`. Use `dataset.metadata` to inspect and list these entities.

In [ ]:
# List available record sets with their @id values
# The dataset metadata exposes record sets under dataset.metadata.recordSet
record_sets = getattr(metadata, 'recordSet', [])

print("Available Record Sets (@id and name):")
record_set_ids = []
for rs in record_sets:
    rs_id = rs.get('@id')
    rs_name = rs.get('name', rs_id)
    record_set_ids.append(rs_id)
    print(f"- {rs_name} (@id: {rs_id})")

# For demonstration, print the available fields in the first record set
if record_sets:
    first_rs = record_sets[0]
    fields = first_rs.get('field', [])
    print("\nFields in first record set:")
    for f in fields:
        print(f"- {f.get('@id')}: {f.get('name', f.get('@id'))}")
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Here, we load all record sets, referencing each by its `@id`.

In [ ]:
# Extract data from each record set
# For demonstration, we load all record sets found.
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        print(f"No records found for record set @id: {record_set_id}")

# Display columns for the first available record set
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Columns in DataFrame for record set @id {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**Reference all fields and columns using their `@id`.**

In [ ]:
import numpy as np

# Example: select a numeric field for analysis from the first dataframe
# Let's search for a likely numeric field (e.g., GAD-7 total score, PHQ-9 score, etc.)
first_rs_id = list(dataframes.keys())[0] if dataframes else None
df = dataframes.get(first_rs_id) if first_rs_id else None

# We'll attempt to find a numeric field in the DataFrame by inspecting columns
numeric_field_id = None
for col in df.columns if df is not None else []:
    if df[col].dtype in [np.float64, np.int64] or (df[col].dtype == object and df[col].apply(lambda x: isinstance(x, (int, float))).any()):
        # Select first numeric field
        numeric_field_id = col
        break

if numeric_field_id:
    threshold = df[numeric_field_id].median() if df[numeric_field_id].dtype in [np.float64, np.int64] else 10
    # Filter for values above threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field (try to pick 'village' or 'gender' if present)
    possible_group_fields = ['village', 'gender', 'marital_status', 'level_of_education']
    group_field_id = None
    for gf in possible_group_fields:
        if gf in df.columns:
            group_field_id = gf
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().rename(columns={numeric_field_id:'mean_score'})
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable group field found in DataFrame.")
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we'll plot the distribution of the chosen numeric field and (if found) show mean scores grouped by a categorical variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution for the numeric field
if df is not None and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping was done, visualize group means
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,4))
        sns.barplot(x=grouped_df[group_field_id], y=grouped_df['mean_score'])
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded dataset metadata and records using the Croissant schema and `mlcroissant`.
- Inspected available record sets and fields by their `@id`.
- Extracted survey data into DataFrames for processing.
- Applied EDA steps such as filtering and normalization of numeric fields, and grouped results by categorical variables.
- Visualized distributions and relationships between demographic and mental health indicators.

This workflow demonstrates the use of the Croissant `@id` referencing and the versatility of the `mlcroissant` library for reproducible data science.